# Primary Model Only — Exercise 3.5b comparison

Split from the original `AFML 3.2.2 - Meta-Labeling with sklearn.ipynb`, which now contains only the meta-labeling (primary + secondary model) exercise.

This notebook is self-contained: it re-reads the data and rebuilds the features before training the primary model *without* meta-labels (`side = None`).

Here we compare (on test data only) the primary-model-only results against the meta-labeling results from the other notebook.

As seen in the labeling exercise, only ~48.04% of the sample was labeled 1.
Hence precision 1.0 = 0.48 (48% of the sample is relevant), while recall = 1 means fully correct (based on the 48% sample).



In [ ]:
import numpy as np
import pandas as pd
import cqrlib as rs
import matplotlib.pyplot as plt

%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

p = print

dollar = pd.read_csv('../sample-data/dollar_bars.csv', 
                 sep=',', 
                 header=0, 
                 parse_dates = True, 
                 index_col=['date_time'])


In [ ]:
# fraction form keeps the original pipeline (dollar) intact; moment form goes to a parallel frame
dollar['ewm'], dollar['upper'], dollar['lower'], dollar['std'] = rs.bband_frac(dollar)

dollar['side'] = np.nan

upper = dollar[dollar['upper'] < dollar['close']] # short signal
lower = dollar[dollar['lower'] > dollar['close']] # long signal

p("Num of times upper limit touched: {0}\nNum of times lower limit touched: {1}"
  .format(len(upper), 
          len(lower)))

# Recall white test as a benchmark and until this stage we filtered all those which did not meet min return
dollar = rs.side_pick(dollar)
dollar.dropna(inplace= True)
dollar['side'].value_counts()


In [ ]:
copy_dollar = dollar.copy() # make a back copy to be used in the feature engineering below

copy_dollar # up till this point the below dataframe should look like this, before tri_bar func. This is our primary model.


In [ ]:
# drop redundant columns and keep crossing moving avaerages
pri_dollar = copy_dollar.drop(['open', 'high', 'low', 'cum_vol', 'cum_dollar', 'cum_ticks'], axis = 1)

# include original volatility
pri_dollar['volatility'] = rs.vol(pri_dollar.close, span0 = 50)

pri_dollar


In [ ]:
# Optional: getting stationarity feature
pri_dollar['log_price'] = pri_dollar.close.apply(np.log)
pri_dollar['log_return'] = pri_dollar.log_price.diff()

cs_log = pri_dollar.log_price.diff().dropna().to_frame()
pri_dollar['stationary'] = rs.fracDiff_FFD(data = cs_log, d = 1.99999889 , thres = 1e-5)

rs.unit_root(pri_dollar['stationary'].dropna()) #check for stationarity


In [ ]:
pri_dollar.dropna(inplace = True)


In [ ]:
# autocorrelation residual feature, we will add AR features up to 2 lags
from statsmodels.tsa.arima.model import ARIMA  # ARMA was removed in statsmodels 0.13

pri_dollar['ar_0'] = ARIMA(pri_dollar['stationary'], order=(0,0,0)).fit().resid
pri_dollar['ar_1'] = ARIMA(pri_dollar['stationary'], order=(1,0,0)).fit().resid
pri_dollar['ar_2'] = ARIMA(pri_dollar['stationary'], order=(2,0,0)).fit().resid


# **Now we start to create only primary model**


In [ ]:
# volatility is a return; cs_filter diffs are price points, so scale by price level
events1 = rs.cs_filter(pri_dollar['close'], 
                    limit = pri_dollar['volatility'].mean() * pri_dollar['close'].mean())

vb1 = rs.vert_barrier(data = pri_dollar['close'], 
                 events = events1, 
                 period = 'days', 
                 freq = 1)

tb1 = rs.tri_barrier(data = pri_dollar['close'], 
                events = events1, 
                trgt = pri_dollar['volatility'], 
                min_req = 0.002, 
                num_threads = 3, 
                ptSl = [0,2], # change ptSl into [0,2]
                t1 = vb1, 
                side = None)

m_label1 = rs.meta_label(data = pri_dollar['close'],
                      events = tb1,
                      drop = 0.05) # take note we do not have a side hence we need to drop something


In [ ]:
# Random Forest Model
n_estimators, max_depth, c_random_state = 100, 5, 42

rf = RandomForestClassifier(max_depth=max_depth, 
                            n_estimators=n_estimators,
                            criterion='entropy', 
                            class_weight = None, #This will be cover in next few chapters
                            random_state=c_random_state)

X = pri_dollar.reindex(m_label1.index) # this dataframe only contain all our features
y = m_label1['bin']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

rf.fit(X_train, y_train.values.ravel())


In [ ]:
# Performance Metrics
y_prob = rf.predict_proba(X_train)[:, 1] #here we are only interested in True positive
y_pred = rf.predict(X_train)

p('Matrix training report for primary model only\n')

rs.report_matrix(actual_data = y_train, # we need to use our train data from train_test_split
                 prediction_data = y_pred, 
                 ROC = y_prob)


In [ ]:
# Meta-label
# Performance Metrics
y_prob = rf.predict_proba(X_test)[:, 1] #here we are only interested in True positive
y_pred = rf.predict(X_test)

p('Matrix test report for primary model only\n')

rs.report_matrix(actual_data = y_test, 
                 prediction_data = y_pred, 
                 ROC = y_prob)

rs.feat_imp(rf, X)


## Based on our matrix report

The comparisons was made in ceteris paribus condition as much as possible.

All comparison will be made based on test data only, train data will be excluded.
    
### Accuracy comparison

**Accuracy is sum of True Positive and True Negative divided by overall set of items**

There is an improvement in accuracy rate when meta-label (Primary & secondary model score yield 0.53), with original yielding only 0.48.

Accuracy +0.05 improvement (10% increase) from the original, +0.0342 improvement (6.8% increase) from primary model only.

However, in order to correctly use this method. Dr Marco Lopez De Prado did mentioned the below:

>"First, we build a model that achieve high recall, even if precision is not particularly high.
>
> Second correct for low precision by applying meta-label to the positives predicted by the primary model." 
>
> Advances in Financial Machine Learning, page 52

However, for primary model only case. (-1,1) are consider price actions label, which in my opinion does not seem to work well with ML. But it does improve accuracy score by a small margin against original data with no labels. 

In our case, we filtered out labels that touched vertical barrier first from primary model only.

### F1 scores comparison
**Measures the efficiency of classifer (Harmonic mean of both precision and recall)**

Using both primary and secondary models to identify True Positive yields a score of 0.6 (for both long and short).

while using primary model only which gives 0.56 (for short) and 0.4 (for long).

F1 score +0.04 (7% increase/ short) and +0.2 (50% increase/ long) improvement, when compared against primary model only.

>"Meta-labeling is particularly helpful when you want to achieve higher F1-scores."
>
> Advances in Financial Machine Learning, page 52

### Other observations

<ins>Stationarity absolute return series</ins> as a optional key feature, does not seem relevant at all since the ML does not recognize it after log price absolute change. Hence it is lowly ranked in feature importance graph.

**For secondary model**

Crossing averages and volatility does seem to be at the top of the key features importance. This seems to say, ML model recognize that as key feature for predictions, while auto-correlation does not seem that important.

The ML model did realised, we were using those indicators as our primary models.

**For Primary model**

The ML model only recognize we were using volatility as our benchmark (tri_barrier func cs_filter as trgt), when we did not use any (0,1) meta-labels.

### Conclusion

To get a higher F1 score and better accuracy, quants should use both primary (To let it decide bet direction) while using a secondary model to decide bet size (To bet or not).

Stationarity is an important concept, especially to mean-reversion strategy. As Stationary series act as an anchor which the strategy will revert to eventually.

We will cover more with other examples regarding stationarity in the next chapter.
